# 01 - Create Control Tables

Creates the Delta control tables used for file-level idempotency,
pipeline audit, incremental watermarks, and schema-change tracking.

In [ ]:
import re

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`control`")

In [ ]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`processed_files` (
        source_name STRING NOT NULL,
        source_file STRING NOT NULL,
        file_size BIGINT,
        modification_time TIMESTAMP,
        content_hash STRING,
        batch_id STRING,
        first_seen_ts TIMESTAMP NOT NULL,
        processed_ts TIMESTAMP,
        pipeline_run_id STRING,
        status STRING NOT NULL,
        rows_read BIGINT,
        rows_written BIGINT,
        rows_quarantined BIGINT,
        error_message STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'File-level ledger used to make Bronze ingestion idempotent'
    )
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`audit_pipeline_runs` (
        pipeline_run_id STRING NOT NULL,
        pipeline_name STRING NOT NULL,
        batch_id STRING,
        start_ts TIMESTAMP NOT NULL,
        end_ts TIMESTAMP,
        status STRING NOT NULL,
        rows_read BIGINT,
        rows_written BIGINT,
        rows_quarantined BIGINT,
        source_paths ARRAY<STRING>,
        notebook_job_run_id STRING,
        error_message STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'One audit record per pipeline execution'
    )
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`pipeline_watermarks` (
        pipeline_name STRING NOT NULL,
        watermark_name STRING NOT NULL,
        watermark_ts TIMESTAMP,
        watermark_value STRING,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Incremental-processing watermarks by pipeline'
    )
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`schema_change_log` (
        source_name STRING NOT NULL,
        detected_ts TIMESTAMP NOT NULL,
        source_file STRING,
        batch_id STRING,
        schema_version STRING NOT NULL,
        previous_schema_json STRING,
        current_schema_json STRING NOT NULL,
        change_type STRING NOT NULL,
        pipeline_run_id STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Detected source schema versions and schema changes'
    )
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`data_quality_results` (
        quality_date DATE NOT NULL,
        source_name STRING NOT NULL,
        pipeline_stage STRING NOT NULL,
        quality_status STRING NOT NULL,
        error_code STRING,
        record_count BIGINT NOT NULL,
        pipeline_run_id STRING NOT NULL,
        recorded_ts TIMESTAMP NOT NULL
    )
    USING DELTA
    PARTITIONED BY (quality_date)
    TBLPROPERTIES (
        'comment' = 'Aggregated valid and error counts used by the DQ dashboard'
    )
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`record_lineage` (
        target_table STRING NOT NULL,
        target_record_key STRING NOT NULL,
        pipeline_run_id STRING NOT NULL,
        source_table STRING NOT NULL,
        source_record_key STRING,
        source_file STRING,
        batch_id STRING,
        source_ingest_ts TIMESTAMP,
        recorded_ts TIMESTAMP NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Record-level lineage from Silver and Gold records back to source files'
    )
    """
)

In [ ]:
for table_name in [
    "processed_files",
    "audit_pipeline_runs",
    "pipeline_watermarks",
    "schema_change_log",
    "data_quality_results",
    "record_lineage",
]:
    print(f"Created or verified: {catalog}.control.{table_name}")